# API Functions → Binary ML Dataset

Downloads `API Functions.csv` directly from a public S3 URL, then flattens it into a binary matrix suitable for machine learning:
- **Rows**: one per sample, keyed by SHA256 as a full integer (`sha256_int`)
- **API columns**: 1 if that API function is present in the sample, 0 otherwise
- **Label columns**: one-hot encoded malware type (Backdoor, Downloader, Generic Malware, Ransomware, Spyware, …)

No file upload or Google Drive required — just run all cells in order.

## Step 1 — Imports

In [ ]:
import io
import os
import requests
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

print(f"pandas {pd.__version__}")

## SHA256 → Integer Helper

In [ ]:
def sha256_to_int(hex_hash: str) -> int:
    """Convert a 64-char hex SHA256 string to its full integer representation."""
    h = str(hex_hash).strip().lower()
    if len(h) != 64:
        raise ValueError(f"Expected 64-char SHA256, got {len(h)} chars: {h!r}")
    return int(h, 16)


def azure_blob_upload_url(container_sas_url: str, blob_name: str) -> str:
    """Build a blob PUT URL from a container-level SAS URL."""
    base, sas = container_sas_url.split("?", 1)
    return f"{base.rstrip('/')}/{blob_name}?{sas}"

## Step 2 — Download CSV from URL

In [ ]:
CSV_URL     = "https://prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com/a59c473d-9076-422a-a8de-952a35591757"
LOCAL_CSV   = "/content/API_Functions.csv"
OUTPUT_FILE = "/content/API_Functions_flat.csv"

AZURE_CONTAINER_SAS_URL = (
    "https://pocstorage1705.blob.core.windows.net/storage"
    "?sp=rw&st=2026-06-09T10:53:58Z&se=2027-06-09T19:08:58Z"
    "&spr=https&sv=2026-02-06&sr=c&sig=LRSJIQT3vHMbeZy52kQjD8DpNoXh5t3NKeDVDP88ujk%3D"
)
OUTPUT_BLOB_NAME = "API_Functions_flat.csv"

if os.path.exists(LOCAL_CSV):
    print(f"Already cached: {LOCAL_CSV}")
else:
    print("Downloading API Functions.csv …")
    with requests.get(CSV_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(LOCAL_CSV, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB chunks
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB", end="\r")
    print(f"\nSaved to {LOCAL_CSV}  ({os.path.getsize(LOCAL_CSV) / 1e6:.1f} MB)")

## Step 3 — Load CSV

Col 0 of the raw CSV is a 64-character hex SHA256 hash. It is converted to a full 256-bit integer via `sha256_to_int()` and stored as the `sha256_int` row index.

In [ ]:
print("Reading CSV …")

df = pd.read_csv(LOCAL_CSV, header=None, index_col=0, low_memory=False)

# Col 0 is the hex SHA256 index; convert to full integer IDs.
df.index = df.index.map(sha256_to_int)
df.index.name = "sha256_int"

# Col 1 (iloc 0 after index_col) → malware type label.
# Cols 2+ (iloc 1+) → API function name columns, padded with empty strings.
label_col = df.iloc[:, 0]
api_cols  = df.iloc[:, 1:]

print(f"Rows loaded  : {len(df):,}")
print(f"Index dtype  : {df.index.dtype}")
print(f"Label counts : {label_col.value_counts().to_dict()}")
print(f"API columns  : {api_cols.shape[1]:,} (including empty padding)")

## Step 4 — Build Binary API Presence Matrix

Each row becomes a **set** of API names (empty strings and literal `"NA"` entries are excluded as padding artifacts). `MultiLabelBinarizer` converts these sets into a 0/1 matrix with one column per unique API name.

In [ ]:
EXCLUDED = {"NA", ""}

def row_to_api_set(row):
    """Return the set of real API names present in this row."""
    return {v for v in row.dropna().astype(str) if v not in EXCLUDED}

print("Parsing API sets per row …")
api_sets = api_cols.apply(row_to_api_set, axis=1)

print("Building binary API presence matrix …")
mlb = MultiLabelBinarizer(sparse_output=False)
api_matrix = pd.DataFrame(
    mlb.fit_transform(api_sets),
    index=df.index,
    columns=mlb.classes_,
    dtype="int8",
)

print(f"API matrix shape : {api_matrix.shape}  "
      f"({api_matrix.shape[0]:,} samples × {api_matrix.shape[1]:,} unique APIs)")
print(f"Memory usage     : {api_matrix.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## Step 5 — One-Hot Encode Malware Labels

In [ ]:
label_matrix = pd.get_dummies(label_col.rename("label"), dtype="int8")
label_matrix.index = df.index

print(f"Label columns : {list(label_matrix.columns)}")
print(label_matrix.sum().rename("sample_count").to_string())

## Step 6 — Concatenate, Save, and Upload

The final CSV has:
- `sha256_int` as the row index (full decimal integer, not hex)
- All unique API function name columns (0/1)
- All malware type columns (0/1) at the right-hand side

It is written locally, then uploaded to Azure Blob Storage as `API_Functions_flat.csv`.

In [ ]:
result = pd.concat([api_matrix, label_matrix], axis=1)

print(f"Final dataset shape : {result.shape}  "
      f"({result.shape[0]:,} rows × {result.shape[1]:,} columns)")
print(f"Total memory        : {result.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\nSaving locally to {OUTPUT_FILE} …")
result.to_csv(OUTPUT_FILE)
local_size_mb = os.path.getsize(OUTPUT_FILE) / 1e6
print(f"Local file size     : {local_size_mb:.1f} MB")

# blob_url = azure_blob_upload_url(AZURE_CONTAINER_SAS_URL, OUTPUT_BLOB_NAME)
# print(f"\nUploading to Azure Blob: {OUTPUT_BLOB_NAME} …")
# with open(OUTPUT_FILE, "rb") as f:
#     upload = requests.put(
#         blob_url,
#         data=f,
#         headers={
#             "x-ms-blob-type": "BlockBlob",
#             "Content-Type": "text/csv",
#         },
#         timeout=600,
#     )
# upload.raise_for_status()
# print(f"Upload complete — blob '{OUTPUT_BLOB_NAME}' in container 'storage'.")

## Step 7 — (Optional) Download Local Copy

If running in Google Colab, download the local copy to your machine.

In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except ImportError:
    print(f"Not in Colab — local file remains at {OUTPUT_FILE}")

## (Optional) Preview the Result

In [ ]:
result.head(3)
print(f"Sample ID : {result.index[0]}  (type: {type(result.index[0]).__name__})")